## PRL-ready Personas Dataset

This notebook prepares `personas.csv` for probabilistic record linkage (PRL).

The goal is to create a clean working dataset where each person mention is enriched with event metadata, normalized names, role groups, evidence flags, and blocking keys.

Input files:
- `personas.csv`
- `bautismos_clean.csv`
- `matrimonios_clean.csv`
- `entierros_clean.csv`

Output files:
- `PRL_personas_ready_full.csv`
- `PRL_ready_personas_final.csv`

In [73]:
# libraries
import numpy as np
import pandas as pd
import re
import unicodedata
import pathlib as Path

## 1. Data Loading
The cleaned datasets are stored in `data/clean` directory and include three types of sacramental records:
 - `bautismos_clean.csv`: Cleaned Baptism records.
 - `entierros_clean.csv`: Cleaned Burial records.
 - `matrimonios_clean.csv`: Cleaned marriage records.
 - `personas.csv`: Cleaned personas data.

In [74]:
BAUTISMOS_CLEAN = pd.read_csv("../data/clean/bautismos_clean.csv")
MATRIMONIOS_CLEAN = pd.read_csv("../data/clean/matrimonios_clean.csv")
ENTIERROS_CLEAN = pd.read_csv("../data/clean/entierros_clean.csv")
PERSONAS_CLEAN= pd.read_csv("../data/clean/personas.csv")

PERSONAS_CLEAN.head(5)

C:\Users\samav\AppData\Local\Temp\ipykernel_5040\4267488176.py:4: DtypeWarning: Columns (13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  PERSONAS_CLEAN= pd.read_csv("../data/clean/personas.csv")


,event_idno,original_identifier,persona_type,name,birth_place,birth_date,legitimacy_status,birth_date_precision,lastname,persona_idno,social_condition,marital_status,resident_in,death_place,death_date,death_date_precision,gender
0,bautizo-1,APAucará-LB-L001_B001,baptized,domingo,NaN,1790-08-04,legitimo,exact,ayquipa,persona-1,NaN,NaN,NaN,NaN,NaN,NaN,male
1,bautizo-1,APAucará-LB-L001_B001,father,lucas,NaN,NaN,NaN,NaN,ayquipa,persona-2,NaN,NaN,NaN,NaN,NaN,NaN,male
2,bautizo-1,APAucará-LB-L001_B001,mother,sevastiana,NaN,NaN,NaN,NaN,quispe,persona-3,NaN,NaN,NaN,NaN,NaN,NaN,female
3,bautizo-1,APAucará-LB-L001_B001,godfather,vicente,NaN,NaN,NaN,NaN,guamani,persona-4,NaN,NaN,NaN,NaN,NaN,NaN,male
4,bautizo-2,APAucará-LB-L001_B002,baptized,dominga,NaN,1790-08-04,legitimo,exact,lulia,persona-5,NaN,NaN,NaN,NaN,NaN,NaN,female


## 2. Creating Event Lookup Table
This function takes an event-level DataFrame and creates a clean lookup table containing only the event information needed for probabilistic record linkage. It recreates a unique original_identifier by combining the file and identifier columns, selects important event-level fields such as `event_type`, `event_date`, and `event_place`, removes duplicate event rows, adds the source table name, and returns the final lookup table.

In [75]:
# function to create an event lookup table
def create_event_lookup(event_df,source_table_name):
    """
    Recreates unique original_identifier by combining the file and identifier column,
    and select the required event level columns.
    """

    event_df = event_df.copy()
    # Recreate original_identifier using file and identifier column.
    event_df['original_identifier'] = (event_df['file'].astype(str)+"_"+event_df['identifier'].astype(str)).str.replace(" ","-",regex=False)

    # Select the event level columns and the original_identifier
    select_columns=[
        "original_identifier","event_type","event_place","event_date"
    ]

    # clean the selected columns in the lookup by finding the duplicated and remove them
    # create a small lookup table so that it can be merged with personas later when returned.

    lookup = event_df[select_columns].drop_duplicates()

    # add the source table to the lookup table

    lookup['source_table']=source_table_name

    return lookup 





In [76]:
# Create the lookup tables for bautismos_clean.csv, matrimonios_clean.csv, entierros_clean.csv
bautismos_lookup = create_event_lookup(BAUTISMOS_CLEAN,"bautismos")
matrimonios_lookup = create_event_lookup(MATRIMONIOS_CLEAN,"matrimonios")
entierros_lookup = create_event_lookup(ENTIERROS_CLEAN,"entierros")

bautismos_lookup.head()

,original_identifier,event_type,event_place,event_date,source_table
0,APAucará-LB-L001_B001,Bautizo,Pampamarca,1790-10-04,bautismos
1,APAucará-LB-L001_B002,Bautizo,Pampamarca,1790-10-06,bautismos
2,APAucará-LB-L001_B003,Bautizo,Pampamarca,1790-10-07,bautismos
3,APAucará-LB-L001_B004,Bautizo,Aucara,1790-10-20,bautismos
4,APAucará-LB-L001_B005,Bautizo,Aucara,1790-10-20,bautismos


In [77]:
# Now combine all 3 lookup tables as event_lookup_tables
event_lookup_table = pd.concat(
    [bautismos_lookup,matrimonios_lookup,entierros_lookup],ignore_index=True
)

event_lookup_table.loc[event_lookup_table['source_table']=='matrimonios'].head()


,original_identifier,event_type,event_place,event_date,source_table
6339,APAucará-LM-L001_M001,Matrimonio,Aucara,1816-12-06,matrimonios
6340,APAucará-LM-L001_M002,Matrimonio,Aucara,1816-12-12,matrimonios
6341,APAucará-LM-L001_M003,Matrimonio,Aucara,1817-03-05,matrimonios
6342,APAucará-LM-L001_M004,Matrimonio,Pampamarca|santa iglesia,1817-03-10,matrimonios
6343,APAucará-LM-L001_M005,Matrimonio,Pampamarca|santa iglesia,1817-03-12,matrimonios


In [78]:
# merge them to personas.csv
personas_ready = PERSONAS_CLEAN.merge(event_lookup_table, on="original_identifier",how="left")
personas_ready.head()

,event_idno,original_identifier,persona_type,name,birth_place,birth_date,legitimacy_status,birth_date_precision,lastname,persona_idno,...,marital_status,resident_in,death_place,death_date,death_date_precision,gender,event_type,event_place,event_date,source_table
0,bautizo-1,APAucará-LB-L001_B001,baptized,domingo,NaN,1790-08-04,legitimo,exact,ayquipa,persona-1,...,NaN,NaN,NaN,NaN,NaN,male,Bautizo,Pampamarca,1790-10-04,bautismos
1,bautizo-1,APAucará-LB-L001_B001,father,lucas,NaN,NaN,NaN,NaN,ayquipa,persona-2,...,NaN,NaN,NaN,NaN,NaN,male,Bautizo,Pampamarca,1790-10-04,bautismos
2,bautizo-1,APAucará-LB-L001_B001,mother,sevastiana,NaN,NaN,NaN,NaN,quispe,persona-3,...,NaN,NaN,NaN,NaN,NaN,female,Bautizo,Pampamarca,1790-10-04,bautismos
3,bautizo-1,APAucará-LB-L001_B001,godfather,vicente,NaN,NaN,NaN,NaN,guamani,persona-4,...,NaN,NaN,NaN,NaN,NaN,male,Bautizo,Pampamarca,1790-10-04,bautismos
4,bautizo-2,APAucará-LB-L001_B002,baptized,dominga,NaN,1790-08-04,legitimo,exact,lulia,persona-5,...,NaN,NaN,NaN,NaN,NaN,female,Bautizo,Pampamarca,1790-10-06,bautismos


In [79]:
# Original Personas shape
print(f"Original Personas Shape: {PERSONAS_CLEAN.shape}")

# merged personas shape
print(f"Merged personas shape: {personas_ready.shape}")

# number of rows with missing event date
print(f"Missing event dates: {personas_ready['event_date'].isna().sum()}")

# number of missing event place
print(f"Missing event place: {personas_ready['event_place'].isna().sum()}")

Original Personas Shape: (47072, 17)
Merged personas shape: (47072, 21)
Missing event dates: 29
Missing event place: 5301


In [80]:
unmatched_rows=personas_ready[personas_ready['source_table'].isna()]
len(unmatched_rows)

0

In [81]:
missing_summary = (
    personas_ready
    .assign(
        missing_event_date=personas_ready["event_date"].isna(),
        missing_event_place=personas_ready["event_place"].isna()
    )
    .groupby(["source_table", "event_type"], dropna=False)
    .agg(
        total_rows=("original_identifier", "size"),
        missing_dates=("missing_event_date", "sum"),
        missing_places=("missing_event_place", "sum")
    )
    .reset_index()
)

missing_summary

,source_table,event_type,total_rows,missing_dates,missing_places
0,bautismos,Bautizo,24986,8,4095
1,entierros,Entierro,5395,11,1196
2,matrimonios,Matrimonio,16691,10,10


In [82]:
# converting event_date, birth_date, death date to datetime
personas_ready['event_date'] = pd.to_datetime(personas_ready['event_date'],errors='coerce')
personas_ready['birth_date'] = pd.to_datetime(personas_ready['birth_date'],errors='coerce')
personas_ready['death_date'] = pd.to_datetime(personas_ready['death_date'],errors='coerce')

# extracting year from the datetime columns
personas_ready['event_year'] = personas_ready['event_date'].dt.year
personas_ready['birth_year'] = personas_ready['birth_date'].dt.year
personas_ready['death_year'] = personas_ready['death_date'].dt.year

In [83]:
# Need to get source_event_type

def source_event_type(eventid_no):
    # Implementation for getting source event type based on event ID

    if pd.isna(eventid_no):
        return "Unknown"
    
    eventid_no = str(eventid_no).lower()

    if eventid_no.startswith("bautizo"):
        return "Baptism"
    elif eventid_no.startswith("matrimonio"):
        return "Marriage"
    elif eventid_no.startswith("entierro"):
        return "Burial"
    else:
        return "unknown"

In [84]:
# Creating the source event type column
personas_ready['source_event_type'] = personas_ready['event_idno'].apply(source_event_type)
personas_ready['source_event_type'].value_counts()

source_event_type
Baptism     24986
Marriage    16691
Burial       5395
Name: count, dtype: int64

## 3. Normalizing the data
This method standardizes text by converting values to lowercase, removing accents, trimming spaces, and handling missing values safely.
It also removes unwanted characters and extra spaces so names/places can be compared consistently during PRL.


In [85]:
def normalize(value):
    if pd.isna(value):
        return "unknown"
    
    # convert the value to a string and make it lowercase and remove the whitespace
    value = str(value).lower().strip()

    # normalize the unicodedata characters by removing accents
    value = unicodedata.normalize('NFD',value)
    value = "".join(char for char in value if unicodedata.category(char)!='Mn')

    # replace the non letter characters with space
    value = re.sub(r"[^a-zñ\s]", " ", value)
    # Replace multiple spaces with one space
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [86]:
# create cleaned firstname
personas_ready['name_clean']= personas_ready['name'].apply(normalize)

# create cleaned lastname
personas_ready['lastname_clean']= personas_ready['lastname'].apply(normalize)

# create cleaned full name
personas_ready['full_name_clean']=(personas_ready['name_clean']+" "+personas_ready['lastname_clean']).str.strip()


In [87]:
# create cleaned birth place
personas_ready['birth_place_clean'] = personas_ready['birth_place'].apply(normalize)

# create cleaned event place
personas_ready['event_place_clean'] = personas_ready['event_place'].apply(normalize)

# create cleaned death place
personas_ready['death_place_clean'] = personas_ready['death_place'].apply(normalize)

# create cleaned residence column
personas_ready['resident_in_clean'] = personas_ready['resident_in'].apply(normalize)


In [88]:
# extracting the first initial from firstname for forming the blockkeys
personas_ready["first_initial"] = (
    personas_ready["name_clean"]
    .str[:1]
    .fillna("")
)

In [89]:
# Extracting first_name cleaned
personas_ready["first_name_clean"] = (
    personas_ready["name_clean"]
    .fillna("")
    .astype(str)
    .str.split()
    .str[0]
    .fillna("")
)

Defining function to simplify the gender value.

In [90]:
def simplify_gender(gender):
    if pd.isna(gender):
        return "unknown"
    
    # convert the gender to string and lower the cases
    gender = str(gender).lower().strip()

    # keep male as male
    if gender == "male":
        return "male"
    
    # keep female as female
    elif gender == "female":
        return "female"

    # keep mostly male as mostly_male
    elif gender == "mostly_male":
        return "mostly_male"
    
    elif gender == "mostly_female":
        return "mostly_female"
    
    else:
        return "unknown"

# apply the gender function on the gender column
personas_ready['gender']=personas_ready['gender'].apply(simplify_gender)


Defining function to group persona roles


In [91]:
def get_personas_roles(persona_type):
    if pd.isna(persona_type):
        return "unknown"

    # convert the persona_type to string and lower the cases
    persona_type = str(persona_type).lower().strip()

    # main_life_event roles
    if persona_type in ["baptized", "husband", "wife", "deceased"]:
        return "main_life_event_person"
    elif persona_type in ["father","mother", "mother_of_wife","mother_of_husband", "father_of_wife", "father_of_husband"]:
        return "family_relation"
    elif "god" in persona_type:
        return "godparent"
    elif persona_type =="witness":
        return "witness"
    else:
        return "other"

# apply the function to the persona_type column
personas_ready['role_group'] = personas_ready['persona_type'].apply(get_personas_roles)

In [92]:
personas_ready.value_counts('gender')

gender
male             20150
female           16393
unknown          10139
mostly_female      288
mostly_male        102
Name: count, dtype: int64

In [93]:
personas_ready.head()

,event_idno,original_identifier,persona_type,name,birth_place,birth_date,legitimacy_status,birth_date_precision,lastname,persona_idno,...,name_clean,lastname_clean,full_name_clean,birth_place_clean,event_place_clean,death_place_clean,resident_in_clean,first_initial,first_name_clean,role_group
0,bautizo-1,APAucará-LB-L001_B001,baptized,domingo,NaN,1790-08-04,legitimo,exact,ayquipa,persona-1,...,domingo,ayquipa,domingo ayquipa,unknown,pampamarca,unknown,unknown,d,domingo,main_life_event_person
1,bautizo-1,APAucará-LB-L001_B001,father,lucas,NaN,NaT,NaN,NaN,ayquipa,persona-2,...,lucas,ayquipa,lucas ayquipa,unknown,pampamarca,unknown,unknown,l,lucas,family_relation
2,bautizo-1,APAucará-LB-L001_B001,mother,sevastiana,NaN,NaT,NaN,NaN,quispe,persona-3,...,sevastiana,quispe,sevastiana quispe,unknown,pampamarca,unknown,unknown,s,sevastiana,family_relation
3,bautizo-1,APAucará-LB-L001_B001,godfather,vicente,NaN,NaT,NaN,NaN,guamani,persona-4,...,vicente,guamani,vicente guamani,unknown,pampamarca,unknown,unknown,v,vicente,godparent
4,bautizo-2,APAucará-LB-L001_B002,baptized,dominga,NaN,1790-08-04,legitimo,exact,lulia,persona-5,...,dominga,lulia,dominga lulia,unknown,pampamarca,unknown,unknown,d,dominga,main_life_event_person


## Creating Blocking Keys

Blocking keys reduce the number of pairwise comparisons by grouping together records that are more likely to refer to the same person.

This notebook creates four blocking keys:

1. `block_lastname_initial`  
   Groups records by cleaned last name and first initial.

2. `block_last4_firstname`  
   Groups records by the first four letters of the cleaned last name and cleaned first name.

3. `block_lastname_birthyear`  
   Groups records by cleaned last name and birth year when birth year is available.

4. `block_lastname_firstname`  
   Groups records by cleaned last name and cleaned first name.

These blocking keys will later be used together through union blocking for candidate-pair generation.

In [94]:
# Blocking Key 1: lastname + first initial
personas_ready['block_lastname_initial'] = (personas_ready['lastname_clean']+"_"+personas_ready['first_initial'])

# Extract first 4 letters of lastname
personas_ready['lastname_first4'] = personas_ready['lastname_clean'].str[:4]

# Blocking Key 2: first four letters of lastname + first name
personas_ready['block_last4_firstname'] = (personas_ready['lastname_first4']+"_"+personas_ready['first_name_clean'])

# converting birth_year column to string safe for blocking key generation
personas_ready['birth_year_BK'] = (personas_ready['birth_year'].apply(lambda x: str(int(x)) if pd.notna(x) else ""))

# Blocking Key 3: lastname + birth year
personas_ready["block_lastname_birthyear"] = np.where(
    personas_ready["birth_year_BK"].fillna("") != "",
    personas_ready["lastname_clean"] + "_" + personas_ready["birth_year_BK"].fillna(""),
    ""
)
# Blocking Key 4: lastname + first name
personas_ready["block_lastname_firstname"] = (personas_ready["lastname_clean"] + "_" + personas_ready["first_name_clean"])        

## 5. Flagging
This step checks how much useful information each persona row has for probabilistic record linkage by creating true/false flags for available fields like name, gender, year, and place.

Then it adds those flags into `matching_evidence_score`, so records with more usable information can be prioritized for stronger matching and records with weak evidence can be handled carefully.




In [95]:
personas_ready["has_name"] = personas_ready["first_name_clean"] != ""
personas_ready["has_lastname"] = personas_ready["lastname_clean"] != ""
personas_ready["has_birth_year"] = personas_ready["birth_year"].notna()
personas_ready["has_death_year"] = personas_ready["death_year"].notna()
personas_ready["has_gender_known"] = personas_ready["gender"].isin(
    ["male", "female","mostly_male","mostly_female"]
)
personas_ready["has_birth_place"] = personas_ready["birth_place_clean"] != ""
personas_ready["has_event_year"] = personas_ready["event_year"].notna()
personas_ready["has_event_place"] = personas_ready["event_place_clean"] != ""





In [96]:
# Fixing unknown place logic as it is causing mismatches
def is_valid_text(value):
    if pd.isna(value):
        return False

    value = str(value).strip().lower()

    if value in ["", "unknown", "nan", "none"]:
        return False

    return True

In [97]:
personas_ready["has_name"] = personas_ready["first_name_clean"].apply(is_valid_text)

personas_ready["has_lastname"] = personas_ready["lastname_clean"].apply(is_valid_text)

personas_ready["has_birth_place"] = personas_ready["birth_place_clean"].apply(is_valid_text)

personas_ready["has_event_place"] = personas_ready["event_place_clean"].apply(is_valid_text)

personas_ready["has_death_place"] = personas_ready["death_place_clean"].apply(is_valid_text)

In [98]:
# Calculate matching evidence score by summing useful evidence flags
personas_ready["matching_evidence_score"] = (
    personas_ready["has_name"].astype(int)
    + personas_ready["has_lastname"].astype(int)
    + personas_ready["has_birth_year"].astype(int)
    + personas_ready["has_death_year"].astype(int)
    + personas_ready["has_gender_known"].astype(int)
    + personas_ready["has_birth_place"].astype(int)
    + personas_ready["has_event_year"].astype(int)
    + personas_ready["has_event_place"].astype(int)
)


Defining a function to convert the evidence score evidence level

In [99]:
def evidence_level(score):

    # High evidence rows
    if score >= 6:
        return "high"

    # Medium evidence rows
    elif score >= 4:
        return "medium"

    # Low evidence rows
    else:
        return "low"

# Apply evidence level logic
personas_ready["matching_evidence_level"] = personas_ready["matching_evidence_score"].apply(
    evidence_level
)

In [100]:
#define function to flag cleaning issues
def create_issue_flags(row):
    issues = []

    if not row["has_name"]:
        issues.append("missing_name")

    if not row["has_lastname"]:
        issues.append("missing_lastname")

    if not is_valid_text(row["block_lastname_initial"]):
        issues.append("missing_primary_block_key")

    if pd.isna(row["event_date"]):
        issues.append("missing_event_date")

    if row["source_event_type"] == "unknown":
        issues.append("unknown_source_event_type")

    if len(issues) == 0:
        return "no_major_issue"

    return "|".join(issues)

# cleaning issue flags
personas_ready['cleaning_issue_flags'] = personas_ready.apply(create_issue_flags, axis=1)


## RESULTS

In [101]:
# Print final row count
print("Rows:", len(personas_ready))

Rows: 47072


In [102]:
# Print final column count
print("Columns:", len(personas_ready.columns))


Columns: 53


In [103]:
# Print source event type counts
print("\nSource event type counts:")
print(personas_ready["source_event_type"].value_counts(dropna=False))



Source event type counts:
source_event_type
Baptism     24986
Marriage    16691
Burial       5395
Name: count, dtype: int64


In [104]:
# Print role group counts
print("\nRole group counts:")
print(personas_ready["role_group"].value_counts(dropna=False))


Role group counts:
role_group
family_relation           20729
main_life_event_person    12571
godparent                  9523
witness                    4249
Name: count, dtype: int64


In [105]:
# Print matching evidence level counts
print("\nEvidence level counts:")
print(personas_ready["matching_evidence_level"].value_counts(dropna=False))


Evidence level counts:
matching_evidence_level
medium    38122
high       7960
low         990
Name: count, dtype: int64


In [106]:
# Print top cleaning issue counts
print("\nCleaning issue counts:")
print(personas_ready["cleaning_issue_flags"].value_counts(dropna=False).head(20))


Cleaning issue counts:
cleaning_issue_flags
no_major_issue                     46664
missing_lastname                     310
missing_name                          69
missing_event_date                    25
missing_name|missing_event_date        4
Name: count, dtype: int64


## EXPORTING CLEANED PRL_READY_PERSONAS_FULL

In [107]:
print("\nFinal PRL Ready Personas:")
# exporting personas_ready to a CSV file
personas_ready.to_csv("../data/interim/PRL_personas_ready_full.csv", index=False)


Final PRL Ready Personas:


In [108]:
# creating the final PRL_Ready_personas.csv which we will actually use for candidate pair generation.

final_prl_columns = [
    "persona_idno",
    "event_idno",
    "original_identifier",
    "source_event_type",
    "source_table",
    "event_type",

    "persona_type",
    "role_group",

    "name",
    "lastname",
    "name_clean",
    "lastname_clean",
    "full_name_clean",
    "first_name_clean",
    "first_initial",

    "event_date",
    "event_year",
    "event_place_clean",

    "birth_date",
    "birth_year",
    "birth_place_clean",

    "death_date",
    "death_year",
    "death_place_clean",

    "resident_in_clean",

    "gender",

    "block_lastname_initial",
    "block_lastname_firstname",
    "block_last4_firstname",
    "block_lastname_birthyear",

    "matching_evidence_level",
    "cleaning_issue_flags"
]

PRL_ready_personas_final = personas_ready[final_prl_columns].copy()

PRL_ready_personas_final.to_csv(
    "../data/interim/PRL_ready_personas_final.csv",
    index=False
)


In [109]:
print("\nBlocking key columns:")
blocking_columns = [
    "block_lastname_initial",
    "block_lastname_firstname",
    "block_last4_firstname",
    "block_lastname_birthyear"
]
print(PRL_ready_personas_final[blocking_columns].head())


Blocking key columns:
  block_lastname_initial block_lastname_firstname block_last4_firstname  \
0              ayquipa_d          ayquipa_domingo          ayqu_domingo   
1              ayquipa_l            ayquipa_lucas            ayqu_lucas   
2               quispe_s        quispe_sevastiana       quis_sevastiana   
3              guamani_v          guamani_vicente          guam_vicente   
4                lulia_d            lulia_dominga          luli_dominga   

  block_lastname_birthyear  
0             ayquipa_1790  
1                           
2                           
3                           
4               lulia_1790  
